In [2]:
%pip install -q numpy scikit-learn pandas requests

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import csv
import json
import time
import hashlib
import logging
import re
import html
import unicodedata
from pathlib import Path
from collections import defaultdict
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import requests

from sklearn.feature_extraction.text import TfidfVectorizer

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger("CDNMF")

API_ROOT = "https://api.artic.edu/api/v1"

OUTPUT_DIR = Path("Database/artic_department_25000")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_PATH = Path("Database/artic_25000") / "artworks_raw.csv"

LIMIT = 25_000
PROCESSED_LIMIT = 21_000
FEATURE_LIMIT = 128
NEIGHBORS = 20

MIN_CLASSES = 6
MAX_CLASSES = 8
MIN_CLASS_SAMPLES = 100
MAX_CLASS_RATIO = 3.0

MIN_PER_ARTIST = 11
MAX_PER_ARTIST = 200
OPTIONAL_FEATURE_COVERAGE = 0.5

LABEL_FIELD = "department_title"

print("Output directory:", OUTPUT_DIR.resolve())


In [ ]:
FIELDS = (
    "id",
    "artist_id",
    "title",
    "artist_title",
    "date_display",
    "place_of_origin",
    "medium_display",
    "department_title",
    "classification_title",
    "category_titles",
    "artwork_type_title",
    "style_title",
    "subject_titles",
    "technique_titles",
    "material_titles",
    "term_titles",
)

TAXONOMY_FIELDS = (
    "classification_titles",
    "category_titles",
    "style_titles",
    "subject_titles",
    "technique_titles",
    "material_titles",
)

CSV_FIELDS = tuple(dict.fromkeys((*FIELDS, *TAXONOMY_FIELDS)))

BASE_FEATURE_FIELDS = (
    "material_titles",
    "technique_titles",
    "place_of_origin",
    "medium_display",
)

OPTIONAL_FEATURE_FIELDS = (
    "subject_titles",
    "style_titles",
)

def as_text(value):
    if isinstance(value, list):
        return " ".join(str(x) for x in value if x)
    if value is None:
        return ""
    return str(value).strip()


def field_value(record, field):
    aliases = {
        "classification_titles": "classification_title",
        "style_titles": "style_title",
    }

    if field in record:
        return record[field]

    alias = aliases.get(field)
    return record.get(alias) if alias else None


def csv_row(record):
    row = {}

    for field in CSV_FIELDS:
        value = field_value(record, field)

        if isinstance(value, (list, dict)):
            row[field] = json.dumps(value, ensure_ascii=False)
        else:
            row[field] = as_text(value)

    return row


def normalize_text(value):
    text = unicodedata.normalize(
        "NFKC",
        html.unescape(as_text(value))
    ).casefold()

    text = re.sub(r"<[^>]*>", " ", text)
    text = re.sub(r"[^\w]+", " ", text, flags=re.UNICODE)

    return " ".join(text.split())


def valid_artwork(record):
    artwork_id = as_text(record.get("id"))

    return (
        artwork_id.isdigit()
        and bool(as_text(record.get("artist_id")))
        and bool(as_text(record.get("title")))
        and bool(as_text(record.get("department_title")))
    )

print("Configured fields:", len(CSV_FIELDS))

In [7]:
def fetch_page(page, limit=100, retries=5):
    params = {
        "fields": ",".join(FIELDS),
        "limit": limit,
        "page": page,
    }

    for attempt in range(retries):
        try:
            response = requests.get(
                f"{API_ROOT}/artworks",
                params=params,
                headers={"User-Agent": "CDNMF-artwork-dataset/1.0"},
                timeout=45,
            )

            if response.status_code == 403:
                raise RuntimeError(
                    "API trả về HTTP 403. Hãy kiểm tra quyền truy cập "
                    "hoặc thử chạy từ mạng khác."
                )

            if response.status_code == 429 or response.status_code >= 500:
                response.raise_for_status()

            response.raise_for_status()
            payload = response.json()

            if not isinstance(payload.get("data"), list):
                raise ValueError("API trả về dữ liệu không hợp lệ.")

            return payload

        except RuntimeError:
            raise

        except (
            requests.RequestException,
            ValueError,
        ) as error:
            if attempt == retries - 1:
                raise RuntimeError(
                    f"Không tải được page {page}: {error}"
                ) from error

            wait = min(2 ** attempt, 30)
            logger.warning(
                "Page %s lỗi, thử lại sau %s giây",
                page,
                wait,
            )
            time.sleep(wait)

    raise RuntimeError(f"Không tải được page {page}")

In [10]:
import requests

TEST_URLS = [
    "https://api.artic.edu/api/v1/artworks?limit=1",
    "https://api.artic.edu/api/v1/artworks/4?fields=id,title",
    "https://www.artic.edu/",
]

for url in TEST_URLS:
    try:
        r = requests.get(
            url,
            headers={
                "AIC-User-Agent": "CDNMF-artwork-dataset/1.0 (educational project)",
                "Accept": "application/json",
            },
            timeout=20,
        )

        print(f"\nURL: {url}")
        print("Status:", r.status_code)
        print("Content-Type:", r.headers.get("Content-Type"))
        print("Response:", r.text[:300].replace("\n", " "))

    except requests.RequestException as e:
        print(f"\nURL: {url}")
        print("Connection error:", repr(e))


URL: https://api.artic.edu/api/v1/artworks?limit=1
Status: 200
Content-Type: application/json
Response: {"pagination":{"total":133169,"limit":1,"offset":0,"total_pages":133169,"current_page":1,"next_url":"https:\/\/api.artic.edu\/api\/v1\/artworks?page=2&limit=1"},"data":[{"id":9021,"api_model":"artworks","api_link":"https:\/\/api.artic.edu\/api\/v1\/artworks\/9021","is_boosted":false,"title":"The Bew

URL: https://api.artic.edu/api/v1/artworks/4?fields=id,title
Status: 200
Content-Type: application/json
Response: {"data":{"id":4,"title":"Priest and Boy"},"info":{"license_text":"The `description` field in this response is licensed under a Creative Commons Attribution 4.0 Generic License (CC-By) and the Terms and Conditions of artic.edu. All other data in this response is licensed under a Creative Commons Zero

URL: https://www.artic.edu/
Status: 200
Content-Type: text/html; charset=UTF-8
Response: <!DOCTYPE html> <html     dir="ltr"     lang="en"     class="no-js s-contrast-header s-e

In [11]:
test_params = {
    "fields": ",".join(FIELDS),
    "limit": 100,
    "page": 1,
}

r = requests.get(
    "https://api.artic.edu/api/v1/artworks",
    params=test_params,
    headers={
        "AIC-User-Agent": "CDNMF-artwork-dataset/1.0",
        "Accept": "application/json",
    },
    timeout=30,
)

print("Status:", r.status_code)
print("URL:", r.url)
print("Response:", r.text[:1000])

if r.ok:
    payload = r.json()
    print("Số bản ghi:", len(payload.get("data", [])))
    print("Trường đầu tiên:", list(payload["data"][0].keys()) if payload.get("data") else [])

Status: 200
URL: https://api.artic.edu/api/v1/artworks?fields=id%2Cartist_id%2Ctitle%2Cartist_title%2Cdate_display%2Cplace_of_origin%2Cmedium_display%2Cdepartment_title%2Cclassification_title%2Ccategory_titles%2Cartwork_type_title%2Cstyle_title%2Csubject_titles%2Ctechnique_titles%2Cmaterial_titles%2Cterm_titles&limit=100&page=1
Response: {"pagination":{"total":133169,"limit":100,"offset":0,"total_pages":1332,"current_page":1,"next_url":"https:\/\/api.artic.edu\/api\/v1\/artworks?page=2&fields=id%2Cartist_id%2Ctitle%2Cartist_title%2Cdate_display%2Cplace_of_origin%2Cmedium_display%2Cdepartment_title%2Cclassification_title%2Ccategory_titles%2Cartwork_type_title%2Cstyle_title%2Csubject_titles%2Ctechnique_titles%2Cmaterial_titles%2Cterm_titles&limit=100"},"data":[{"id":9021,"title":"The Bewitched Mill","date_display":"1913","place_of_origin":"Germany","medium_display":"Oil on canvas","artwork_type_title":"Painting","department_title":"Modern Art","artist_id":16157,"artist_title":"Franz Marc

In [12]:
def fetch_page(page, limit=100, retries=5):
    params = {
        "fields": ",".join(FIELDS),
        "limit": limit,
        "page": page,
    }

    headers = {
        "AIC-User-Agent": "CDNMF-artwork-dataset/1.0",
        "Accept": "application/json",
    }

    for attempt in range(retries):
        try:
            response = requests.get(
                f"{API_ROOT}/artworks",
                params=params,
                headers=headers,
                timeout=45,
            )

            if response.status_code == 403:
                print("URL bị từ chối:", response.url)
                print("Nội dung:", response.text[:500])
                raise RuntimeError(
                    "HTTP 403: máy chủ từ chối yêu cầu. "
                    "Kiểm tra URL và phản hồi ở trên."
                )

            if response.status_code == 429 or response.status_code >= 500:
                if attempt == retries - 1:
                    response.raise_for_status()

                wait = min(2 ** attempt, 30)
                print(f"HTTP {response.status_code}; đợi {wait}s")
                time.sleep(wait)
                continue

            response.raise_for_status()
            payload = response.json()

            if not isinstance(payload, dict) or not isinstance(
                payload.get("data"), list
            ):
                raise ValueError("API response không có danh sách data hợp lệ.")

            return payload

        except RuntimeError:
            raise

        except (requests.RequestException, ValueError) as error:
            if attempt == retries - 1:
                raise RuntimeError(
                    f"Không tải được page {page}: {error}"
                ) from error

            wait = min(2 ** attempt, 30)
            print(f"Lỗi: {error}. Thử lại sau {wait}s")
            time.sleep(wait)

    raise RuntimeError(f"Không tải được page {page}")

In [13]:
def crawl_artworks(limit=LIMIT, raw_path=RAW_PATH):
    raw_path.parent.mkdir(parents=True, exist_ok=True)

    checkpoint_path = raw_path.with_suffix(".checkpoint.json")

    if raw_path.exists() and raw_path.stat().st_size > 0:
        existing_df = pd.read_csv(
            raw_path,
            dtype=str,
            keep_default_na=False,
        )

        if list(existing_df.columns) != list(CSV_FIELDS):
            raise ValueError("Header CSV không khớp cấu hình hiện tại.")

        seen_ids = set(existing_df["id"])
    else:
        seen_ids = set()

    checkpoint = None

    if checkpoint_path.exists():
        with checkpoint_path.open(encoding="utf-8") as f:
            checkpoint = json.load(f)

    if checkpoint and checkpoint.get("target") != limit:
        if not checkpoint.get("completed", False):
            raise ValueError(
                "LIMIT khác checkpoint cũ. Hãy dùng lại LIMIT cũ "
                "hoặc chuyển CSV/checkpoint cũ sang thư mục khác."
            )

    if not checkpoint and seen_ids:
        raise ValueError(
            "CSV đã có dữ liệu nhưng không có checkpoint. "
            "Hãy khôi phục checkpoint hoặc dùng file mới."
        )

    if checkpoint and checkpoint.get("completed"):
        if len(seen_ids) >= limit:
            print(f"Dataset đã crawl đủ {len(seen_ids)} artworks.")
            return len(seen_ids)

    page = int(checkpoint.get("next_page", 1)) if checkpoint else 1
    scanned = int(checkpoint.get("scanned", 0)) if checkpoint else 0

    mode = "a" if raw_path.exists() and raw_path.stat().st_size > 0 else "w"

    with raw_path.open(
        mode,
        encoding="utf-8-sig",
        newline="",
    ) as f:

        writer = csv.DictWriter(f, fieldnames=CSV_FIELDS)

        if mode == "w":
            writer.writeheader()
            f.flush()

        while len(seen_ids) < limit:
            payload = fetch_page(page, limit=100)
            records = payload["data"]

            if not records:
                break

            for record in records:
                scanned += 1

                if not valid_artwork(record):
                    continue

                artwork_id = as_text(record["id"])

                if artwork_id in seen_ids:
                    continue

                writer.writerow(csv_row(record))
                seen_ids.add(artwork_id)

                if len(seen_ids) >= limit:
                    break

            f.flush()

            pagination = payload.get("pagination") or {}
            total_pages = pagination.get("total_pages")

            checkpoint_state = {
                "target": limit,
                "next_page": page if len(seen_ids) >= limit else page + 1,
                "retained": len(seen_ids),
                "scanned": scanned,
                "completed": (
                    len(seen_ids) >= limit
                    or len(records) < 100
                    or (total_pages and page >= total_pages)
                ),
            }

            temp_path = checkpoint_path.with_suffix(".tmp")
            temp_path.write_text(
                json.dumps(checkpoint_state, indent=2),
                encoding="utf-8",
            )
            temp_path.replace(checkpoint_path)

            print(
                f"Page {page} | Scanned: {scanned:,} | "
                f"Saved: {len(seen_ids):,}/{limit:,}"
            )

            if checkpoint_state["completed"]:
                break

            page += 1
            time.sleep(0.15)

    return len(seen_ids)


downloaded = crawl_artworks()

print(f"\nĐã lưu {downloaded:,} artworks")
print("CSV:", RAW_PATH.resolve())

Page 1 | Scanned: 100 | Saved: 92/25,000
Page 2 | Scanned: 200 | Saved: 192/25,000
Page 3 | Scanned: 300 | Saved: 272/25,000
Page 4 | Scanned: 400 | Saved: 359/25,000
Page 5 | Scanned: 500 | Saved: 426/25,000
Page 6 | Scanned: 600 | Saved: 508/25,000
Page 7 | Scanned: 700 | Saved: 608/25,000
Page 8 | Scanned: 800 | Saved: 708/25,000
Page 9 | Scanned: 900 | Saved: 794/25,000
Page 10 | Scanned: 1,000 | Saved: 794/25,000
Page 11 | Scanned: 1,100 | Saved: 835/25,000
Page 12 | Scanned: 1,200 | Saved: 874/25,000
Page 13 | Scanned: 1,300 | Saved: 887/25,000
Page 14 | Scanned: 1,400 | Saved: 894/25,000
Page 15 | Scanned: 1,500 | Saved: 918/25,000
Page 16 | Scanned: 1,600 | Saved: 981/25,000
Page 17 | Scanned: 1,700 | Saved: 1,061/25,000
Page 18 | Scanned: 1,800 | Saved: 1,142/25,000
Page 19 | Scanned: 1,900 | Saved: 1,240/25,000
Page 20 | Scanned: 2,000 | Saved: 1,340/25,000
Page 21 | Scanned: 2,100 | Saved: 1,397/25,000
Page 22 | Scanned: 2,200 | Saved: 1,476/25,000
Page 23 | Scanned: 2,300 |

In [14]:

import pandas as pd
import numpy as np
import os
import json
import ast
import re
import matplotlib.pyplot as plt
import seaborn as sns

# Đường dẫn dữ liệu
DATA_PATH = "/kaggle/working/Database/artic_25000/artworks_raw.csv"

df = pd.read_csv(DATA_PATH, low_memory=False)

print("=" * 60)
print("1. TỔNG QUAN DATASET")
print("=" * 60)

print(f"Số dòng: {len(df):,}")
print(f"Số cột: {len(df.columns)}")
print(f"Dung lượng RAM: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("\nDanh sách cột:")
print(df.columns.tolist())

print("\nKiểu dữ liệu:")
print(df.dtypes)

print("\n5 dòng đầu:")
display(df.head())

print("\nSố ID tác phẩm duy nhất:")
print(df["id"].nunique())

print("\nSố ID bị trùng:")
print(df["id"].duplicated().sum())

print("\nSố tác phẩm trùng toàn bộ dòng:")
print(df.duplicated().sum())

1. TỔNG QUAN DATASET
Số dòng: 25,000
Số cột: 18
Dung lượng RAM: 28.06 MB

Danh sách cột:
['id', 'artist_id', 'title', 'artist_title', 'date_display', 'place_of_origin', 'medium_display', 'department_title', 'classification_title', 'category_titles', 'artwork_type_title', 'style_title', 'subject_titles', 'technique_titles', 'material_titles', 'term_titles', 'classification_titles', 'style_titles']

Kiểu dữ liệu:
id                        int64
artist_id                 int64
title                    object
artist_title             object
date_display             object
place_of_origin          object
medium_display           object
department_title         object
classification_title     object
category_titles          object
artwork_type_title       object
style_title              object
subject_titles           object
technique_titles         object
material_titles          object
term_titles              object
classification_titles    object
style_titles             object
dtype: ob

,id,artist_id,title,artist_title,date_display,place_of_origin,medium_display,department_title,classification_title,category_titles,artwork_type_title,style_title,subject_titles,technique_titles,material_titles,term_titles,classification_titles,style_titles
0,9021,16157,The Bewitched Mill,Franz Marc,1913,Germany,Oil on canvas,Modern Art,oil on canvas,"[""Modern Art""]",Painting,NaN,"[""Century of Progress"", ""world's fairs"", ""Chic...",[],[],"[""oil on canvas"", ""Century of Progress"", ""pain...",oil on canvas,NaN
1,2848,37279,Scene from Bohemian Life,James McNeill Whistler,1855/57,United States,Pen and black ink and black watercolor over gr...,Prints and Drawings,pen and ink drawings,"[""Prints and Drawings""]",Drawing and Watercolor,NaN,[],[],"[""paper (fiber product)"", ""ink"", ""wash"", ""grap...","[""pen and ink drawings"", ""paper (fiber product...",pen and ink drawings,NaN
2,14837,34988,Netting the Fish,Winslow Homer,1889,United States,"Transparent watercolor, heightened with opaque...",Prints and Drawings,ink or chalk wash,"[""Winslow Homer: Behind the Scenes"", ""Prints a...",Drawing and Watercolor,NaN,"[""boats"", ""animals"", ""people"", ""fishing"", ""wil...","[""scraping"", ""aquarelle""]","[""paper (fiber product)"", ""wash"", ""coating (ma...","[""ink or chalk wash"", ""boats"", ""paper (fiber p...",ink or chalk wash,NaN
3,16571,35809,"Arrival of the Normandy Train, Gare Saint-Lazare",Claude Monet,1877,France,Oil on canvas,Painting and Sculpture of Europe,painting,"[""Painting and Sculpture of Europe""]",Painting,Impressionism,"[""landscapes"", ""travel"", ""trains"", ""train stat...","[""oil painting"", ""painting techniques"", ""paint...","[""oil paint (paint)"", ""painting"", ""paint"", ""ca...","[""painting"", ""Impressionism"", ""landscapes"", ""o...",painting,Impressionism
4,24836,35480,Brushstroke with Spatter,Roy Lichtenstein,1966,United States,Oil and Magna on canvas,Contemporary Art,painting,"[""Contemporary Art"", ""Roy Lichtenstein Retrosp...",Painting,Pop Art,"[""abstract shapes"", ""yellow (color)"", ""blue (c...","[""oil painting"", ""painting (image making)""]","[""oil paint (paint)"", ""magna"", ""canvas""]","[""painting"", ""Pop Art"", ""abstract shapes"", ""oi...",painting,Pop Art



Số ID tác phẩm duy nhất:
25000

Số ID bị trùng:
0

Số tác phẩm trùng toàn bộ dòng:
0


kiem tra


In [ ]:

import pandas as pd
import json
import ast
import re
import unicodedata

# 1. Đọc dữ liệu
DATA_PATH = RAW_PATH
df = pd.read_csv(DATA_PATH, low_memory=False)


# 2. Hàm tiền xử lý văn bản thông thường
def preprocess_text(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # Chuẩn hóa Unicode
    text = unicodedata.normalize("NFKC", text)

    # Chuẩn hóa khoảng trắng
    text = text.replace("\u00a0", " ")
    text = re.sub(r"\s+", " ", text).strip()

    # Không chuyển toàn bộ thành chữ thường
    # Giữ nguyên chữ hoa, chữ thường và dấu câu
    return text


# 3. Hàm xử lý các cột danh sách
def preprocess_list_text(value):
    if pd.isna(value):
        return ""

    if isinstance(value, (list, tuple)):
        items = value
    else:
        value = str(value).strip()

        if not value:
            return ""

        # Đọc chuỗi dạng JSON hoặc Python list
        try:
            items = json.loads(value)
        except (json.JSONDecodeError, TypeError):
            try:
                items = ast.literal_eval(value)
            except (ValueError, SyntaxError):
                return preprocess_text(value)

    if not isinstance(items, (list, tuple)):
        items = [items]

    cleaned_items = []

    for item in items:
        if item is None:
            continue

        cleaned = preprocess_text(item)

        if cleaned:
            cleaned_items.append(cleaned)

    # Giữ dấu phẩy phân cách các chủ đề
    return ", ".join(cleaned_items)


# 4. Các cột văn bản thông thường
text_columns = [
    "title",
    "artist_title",
    "date_display",
    "place_of_origin",
    "medium_display",
    "department_title",
    "classification_title",
    "artwork_type_title",
    "style_title"
]

# 5. Các cột chứa danh sách
list_columns = [
    "category_titles",
    "subject_titles",
    "technique_titles",
    "material_titles",
    "term_titles",
    "classification_titles",
    "style_titles"
]

# 6. Tiền xử lý, tạo cột mới có hậu tố _clean
for col in text_columns:
    if col in df.columns:
        df[col + "_clean"] = df[col].map(preprocess_text)

for col in list_columns:
    if col in df.columns:
        df[col + "_clean"] = df[col].map(preprocess_list_text)

# 7. Hiển thị kết quả
print("Kích thước dữ liệu:", df.shape)

for col in [
    "title",
    "artist_title",
    "subject_titles",
    "category_titles",
    "department_title"
]:
    clean_col = col + "_clean"

    if col in df.columns and clean_col in df.columns:
        print(f"\n--- {col} ---")
        display(df[[col, clean_col]].head(5))

print("\nHoàn tất tiền xử lý văn bản!")


In [27]:
def read_artwork_csv(path, limit=None):
    json_fields = set(TAXONOMY_FIELDS) | {"term_titles"}

    with open(path, encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)

        for row_number, row in enumerate(reader, start=1):
            if limit is not None and row_number > limit:
                break

            record = dict(row)

            for field in json_fields:
                value = record.get(field, "")

                if value:
                    try:
                        record[field] = json.loads(value)
                    except json.JSONDecodeError:
                        record[field] = [value]

            for normalized, original in (
                ("classification_titles", "classification_title"),
                ("style_titles", "style_title"),
            ):
                if not record.get(normalized) and record.get(original):
                    record[normalized] = record[original]

            yield record


records = list(read_artwork_csv(RAW_PATH, limit=LIMIT))

print("Số bản ghi đọc được:", len(records))
print("Số bản ghi hợp lệ:", sum(valid_artwork(r) for r in records))

pd.DataFrame([csv_row(r) for r in records[:5]])

Số bản ghi đọc được: 25000
Số bản ghi hợp lệ: 25000


,id,artist_id,title,artist_title,date_display,place_of_origin,medium_display,department_title,classification_title,category_titles,artwork_type_title,style_title,subject_titles,technique_titles,material_titles,term_titles,classification_titles,style_titles
0,9021,16157,The Bewitched Mill,Franz Marc,1913,Germany,Oil on canvas,Modern Art,oil on canvas,"[""Modern Art""]",Painting,,"[""Century of Progress"", ""world's fairs"", ""Chic...",[],[],"[""oil on canvas"", ""Century of Progress"", ""pain...","[""oil on canvas""]",
1,2848,37279,Scene from Bohemian Life,James McNeill Whistler,1855/57,United States,Pen and black ink and black watercolor over gr...,Prints and Drawings,pen and ink drawings,"[""Prints and Drawings""]",Drawing and Watercolor,,[],[],"[""paper (fiber product)"", ""ink"", ""wash"", ""grap...","[""pen and ink drawings"", ""paper (fiber product...","[""pen and ink drawings""]",
2,14837,34988,Netting the Fish,Winslow Homer,1889,United States,"Transparent watercolor, heightened with opaque...",Prints and Drawings,ink or chalk wash,"[""Winslow Homer: Behind the Scenes"", ""Prints a...",Drawing and Watercolor,,"[""boats"", ""animals"", ""people"", ""fishing"", ""wil...","[""scraping"", ""aquarelle""]","[""paper (fiber product)"", ""wash"", ""coating (ma...","[""ink or chalk wash"", ""boats"", ""paper (fiber p...","[""ink or chalk wash""]",
3,16571,35809,"Arrival of the Normandy Train, Gare Saint-Lazare",Claude Monet,1877,France,Oil on canvas,Painting and Sculpture of Europe,painting,"[""Painting and Sculpture of Europe""]",Painting,Impressionism,"[""landscapes"", ""travel"", ""trains"", ""train stat...","[""oil painting"", ""painting techniques"", ""paint...","[""oil paint (paint)"", ""painting"", ""paint"", ""ca...","[""painting"", ""Impressionism"", ""landscapes"", ""o...","[""painting""]","[""Impressionism""]"
4,24836,35480,Brushstroke with Spatter,Roy Lichtenstein,1966,United States,Oil and Magna on canvas,Contemporary Art,painting,"[""Contemporary Art"", ""Roy Lichtenstein Retrosp...",Painting,Pop Art,"[""abstract shapes"", ""yellow (color)"", ""blue (c...","[""oil painting"", ""painting (image making)""]","[""oil paint (paint)"", ""magna"", ""canvas""]","[""painting"", ""Pop Art"", ""abstract shapes"", ""oi...","[""painting""]","[""Pop Art""]"


In [ ]:
def stable_record_key(record):
    return hashlib.sha256(
        as_text(record.get("id")).encode("utf-8")
    ).digest()


def prune_small_artists(records, min_per_artist):
    counts = defaultdict(int)

    for record in records:
        counts[as_text(record["artist_id"])] += 1

    valid_artists = {
        artist_id
        for artist_id, count in counts.items()
        if count >= min_per_artist
    }

    return [
        record for record in records
        if as_text(record["artist_id"]) in valid_artists
    ]


def balance_records(
    records,
    label_field="department_title",
    min_classes=MIN_CLASSES,
    max_classes=MAX_CLASSES,
    min_class_samples=MIN_CLASS_SAMPLES,
    max_class_ratio=MAX_CLASS_RATIO,
    max_per_artist=MAX_PER_ARTIST,
    min_per_artist=MIN_PER_ARTIST,
):
    by_artist = defaultdict(list)

    for record in records:
        by_artist[as_text(record["artist_id"])].append(record)

    capped = []

    for artist_records in by_artist.values():
        capped.extend(
            sorted(artist_records, key=stable_record_key)[:max_per_artist]
        )

    capped = prune_small_artists(capped, min_per_artist)

    counts = defaultdict(int)

    for record in capped:
        counts[as_text(record[label_field])] += 1

    selected_labels = {
        label
        for label, count in sorted(
            counts.items(),
            key=lambda item: (-item[1], item[0].casefold()),
        )
        if count >= min_class_samples
    }

    selected_labels = set(
        sorted(
            selected_labels,
            key=lambda label: (-counts[label], label.casefold()),
        )[:max_classes]
    )

    if len(selected_labels) < min_classes:
        raise ValueError(
            f"Chỉ tìm thấy {len(selected_labels)} lớp đủ mẫu; "
            f"cần ít nhất {min_classes}. "
            "Hãy tăng LIMIT hoặc giảm MIN_CLASS_SAMPLES."
        )

    balanced = [
        record for record in capped
        if as_text(record[label_field]) in selected_labels
    ]

    balanced = prune_small_artists(balanced, min_per_artist)

    while True:
        counts = defaultdict(int)

        for record in balanced:
            counts[as_text(record[label_field])] += 1

        small_labels = {
            label for label, count in counts.items()
            if count < min_class_samples
        }

        if small_labels:
            balanced = [
                record for record in balanced
                if as_text(record[label_field]) not in small_labels
            ]
            balanced = prune_small_artists(balanced, min_per_artist)
            continue

        if len(counts) < min_classes:
            raise ValueError(
                "Không đủ số lớp sau bước cân bằng. "
                "Hãy tăng dữ liệu đầu vào hoặc điều chỉnh ngưỡng."
            )

        smallest = min(counts.values())
        largest_label, largest_count = min(
            counts.items(),
            key=lambda item: (-item[1], item[0].casefold()),
        )

        target = int(max_class_ratio * smallest)

        if largest_count <= target:
            return balanced

        artist_rows = defaultdict(list)

        for record in balanced:
            artist_rows[as_text(record["artist_id"])].append(record)

        candidates = sorted(
            [
                (artist_id, rows)
                for artist_id, rows in artist_rows.items()
                if any(
                    as_text(row[label_field]) == largest_label
                    for row in rows
                )
            ],
            key=lambda item: (
                -sum(
                    as_text(row[label_field]) == largest_label
                    for row in item[1]
                ),
                item[0],
            ),
        )

        removed = False

        for artist_id, rows in candidates:
            contributions = defaultdict(int)

            for row in rows:
                contributions[as_text(row[label_field])] += 1

            if (
                largest_count - contributions[largest_label]
                < min_class_samples
            ):
                continue

            if any(
                count - contributions.get(label, 0) < min_class_samples
                for label, count in counts.items()
                if label != largest_label
            ):
                continue

            balanced = [
                record for record in balanced
                if as_text(record["artist_id"]) != artist_id
            ]

            balanced = prune_small_artists(
                balanced,
                min_per_artist,
            )

            removed = True
            break

        if removed:
            continue

        # Không thể loại toàn bộ nghệ sĩ mà không làm mất lớp.
        # Giảm riêng số mẫu của lớp lớn nhất theo thứ tự xác định.
        artist_id, rows = candidates[0]

        target_rows = sorted(
            [
                row for row in rows
                if as_text(row[label_field]) == largest_label
            ],
            key=stable_record_key,
        )

        removable = min(
            len(target_rows),
            len(rows) - min_per_artist,
        )

        if removable <= 0:
            raise ValueError(
                "Không thể cân bằng lớp mà vẫn giữ các giới hạn nghệ sĩ."
            )

        remove_count = min(largest_count - target, removable)
        remove_ids = {
            as_text(row["id"])
            for row in target_rows[:remove_count]
        }

        balanced = [
            record for record in balanced
            if as_text(record["id"]) not in remove_ids
        ]

        balanced = prune_small_artists(
            balanced,
            min_per_artist,
        )

In [ ]:
def prepare_dataset(
    records,
    output_dir=OUTPUT_DIR,
    processed_limit=PROCESSED_LIMIT,
    feature_limit=FEATURE_LIMIT,
    neighbors=NEIGHBORS,
    label_field=LABEL_FIELD,
):
    usable = []
    seen_ids = set()
    dropped = 0

    for record in records:
        artwork_id = as_text(record.get("id"))
        label = as_text(record.get(label_field))

        if not valid_artwork(record) or not label:
            dropped += 1
            continue

        if artwork_id in seen_ids:
            dropped += 1
            continue

        seen_ids.add(artwork_id)
        usable.append(record)

    # Chọn tập con tái lập được.
    if len(usable) > processed_limit:
        usable = sorted(
            usable,
            key=stable_record_key,
        )[:processed_limit]

    selected = balance_records(usable, label_field=label_field)

    if len(selected) < 2:
        raise ValueError("Không đủ tác phẩm để tạo graph.")

    labels = sorted({
        as_text(record[label_field])
        for record in selected
    })

    label_ids = {
        label: index
        for index, label in enumerate(labels)
    }

    # Chọn trường tùy chọn có độ phủ cao nhất.
    coverage = {
        field: sum(
            bool(as_text(field_value(record, field)))
            for record in selected
        ) / len(selected)
        for field in OPTIONAL_FEATURE_FIELDS
    }

    optional_field = max(
        OPTIONAL_FEATURE_FIELDS,
        key=lambda field: (
            coverage[field],
            field == "subject_titles",
        ),
    )

    feature_fields = BASE_FEATURE_FIELDS

    if coverage[optional_field] >= OPTIONAL_FEATURE_COVERAGE:
        feature_fields += (optional_field,)

    texts = [
        normalize_text(
            " ".join(
                as_text(field_value(record, field))
                for field in feature_fields
            )
        )
        for record in selected
    ]

    vectorizer = TfidfVectorizer(
        max_features=feature_limit,
        min_df=2,
        ngram_range=(1, 2),
        stop_words="english",
        sublinear_tf=True,
        dtype=np.float32,
    )

    features = vectorizer.fit_transform(texts).tocsr()

    if features.shape[1] == 0:
        raise ValueError("Không tạo được feature TF-IDF.")

    node_ids = [
        as_text(record["id"])
        for record in selected
    ]

    # Graph vô hướng: mỗi cặp node chỉ xuất hiện một lần.
    edge_weights = {}
    artist_nodes = defaultdict(list)

    for index, record in enumerate(selected):
        artist_nodes[as_text(record["artist_id"])].append(index)

    for indices in artist_nodes.values():
        n = len(indices)

        if n < 2:
            continue

        # Mỗi node nối tối đa khoảng neighbors node theo vòng tròn.
        offsets = range(1, min(neighbors // 2, (n - 1) // 2) + 1)

        for offset in offsets:
            for position, source in enumerate(indices):
                target = indices[(position + offset) % n]

                if source == target:
                    continue

                edge = (min(source, target), max(source, target))
                edge_weights[edge] = 1.0

        # Khi số node chẵn và neighbors lẻ, thêm cạnh đối diện.
        if neighbors % 2 == 1 and n > 2 and n % 2 == 0:
            offset = n // 2

            for position, source in enumerate(indices):
                target = indices[(position + offset) % n]
                edge_weights[
                    (min(source, target), max(source, target))
                ] = 1.0

    output_dir.mkdir(parents=True, exist_ok=True)

    # features.txt
    with (output_dir / "features.txt").open(
        "w", encoding="utf-8"
    ) as f:
        for index, artwork_id in enumerate(node_ids):
            values = features.getrow(index).toarray().ravel()

            f.write(
                artwork_id + " " +
                " ".join(format(float(v), ".7g") for v in values) +
                "\n"
            )

    # edges.txt: artwork ID nguồn, artwork ID đích, trọng số
    with (output_dir / "edges.txt").open(
        "w", encoding="utf-8"
    ) as f:
        for (source, target), weight in sorted(edge_weights.items()):
            f.write(
                f"{node_ids[source]} {node_ids[target]} {weight:.7g}\n"
            )

    # nodes.csv
    node_rows = []

    for index, record in enumerate(selected):
        node_rows.append({
            "node_index": index,
            "artwork_id": node_ids[index],
            "artist_id": as_text(record.get("artist_id")),
            "title": as_text(record.get("title")),
            "artist_title": as_text(record.get("artist_title")),
            "label_id": label_ids[as_text(record[label_field])],
            "department_title": as_text(record.get("department_title")),
            "artwork_type_title": as_text(record.get("artwork_type_title")),
        })

    pd.DataFrame(node_rows).to_csv(
        output_dir / "nodes.csv",
        index=False,
        encoding="utf-8-sig",
    )

    # edges.csv
    edge_rows = [
        {
            "source_node_index": source,
            "target_node_index": target,
            "source_artwork_id": node_ids[source],
            "target_artwork_id": node_ids[target],
            "weight": weight,
        }
        for (source, target), weight in sorted(edge_weights.items())
    ]

    pd.DataFrame(
        edge_rows,
        columns=[
            "source_node_index",
            "target_node_index",
            "source_artwork_id",
            "target_artwork_id",
            "weight",
        ],
    ).to_csv(
        output_dir / "edges.csv",
        index=False,
        encoding="utf-8-sig",
    )

    # group.txt
    with (output_dir / "group.txt").open(
        "w", encoding="utf-8"
    ) as f:
        for record, artwork_id in zip(selected, node_ids):
            label_id = label_ids[as_text(record[label_field])]
            f.write(f"{artwork_id} {label_id}\n")

    # walk matrix dạng cạnh, theo quy ước của code gốc.
    (output_dir / "walks.txt").write_text(
        (output_dir / "edges.txt").read_text(encoding="utf-8"),
        encoding="utf-8",
    )

    # Mapping nhãn
    pd.DataFrame([
        {
            "label_id": index,
            label_field: label,
        }
        for label, index in label_ids.items()
    ]).to_csv(
        output_dir / "label_mapping.csv",
        index=False,
        encoding="utf-8-sig",
    )

    # Artwork metadata
    artwork_rows = []

    for record, normalized_text in zip(selected, texts):
        row = csv_row(record)
        row["normalized_text"] = normalized_text
        artwork_rows.append(row)

    pd.DataFrame(artwork_rows).to_csv(
        output_dir / "artworks.csv",
        index=False,
        encoding="utf-8-sig",
    )

    label_counts = defaultdict(int)
    artist_counts = defaultdict(int)

    for record in selected:
        label_counts[as_text(record[label_field])] += 1
        artist_counts[as_text(record["artist_id"])] += 1

    result = {
        "processed_artworks": len(selected),
        "dropped_records": dropped,
        "classes": len(labels),
        "features": int(features.shape[1]),
        "edges": len(edge_weights),
    }

    metadata = {
        **result,
        "label_field": label_field,
        "feature_fields": list(feature_fields),
        "optional_feature_coverage": coverage,
        "class_counts": dict(label_counts),
        "artist_limits": {
            "min_artworks": MIN_PER_ARTIST,
            "max_artworks": MAX_PER_ARTIST,
            "largest_artist_count": max(artist_counts.values()),
        },
        "graph_strategy": "same_artist_cyclic_neighbors",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
    }

    (output_dir / "metadata.json").write_text(
        json.dumps(metadata, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    return result

In [ ]:
result = prepare_dataset(
    records=records,
    label_field=LABEL_FIELD,
    output_dir=OUTPUT_DIR,
    processed_limit=PROCESSED_LIMIT,
    feature_limit=FEATURE_LIMIT,
    neighbors=NEIGHBORS,
)

print(json.dumps(result, indent=2, ensure_ascii=False))

In [31]:
expected_files = [
    "artworks_raw.csv",
    "features.txt",
    "edges.txt",
    "nodes.csv",
    "edges.csv",
    "group.txt",
    "walks.txt",
    "label_mapping.csv",
    "artworks.csv",
    "metadata.json",
]

for filename in expected_files:
    path = OUTPUT_DIR / filename

    if path.exists():
        print(f"OK  {filename:25s} {path.stat().st_size:,} bytes")
    else:
        print(f"MISS {filename}")

print("\nMetadata:")
with (OUTPUT_DIR / "metadata.json").open(encoding="utf-8") as f:
    metadata = json.load(f)

print(json.dumps(metadata, indent=2, ensure_ascii=False))

OK  artworks_raw.csv          11,131,953 bytes
OK  features.txt              754,072 bytes
OK  edges.txt                 264,804 bytes
OK  nodes.csv                 178,930 bytes
OK  edges.csv                 450,658 bytes
OK  group.txt                 18,693 bytes
OK  walks.txt                 264,804 bytes
OK  label_mapping.csv         142 bytes
OK  artworks.csv              1,177,882 bytes
OK  metadata.json             817 bytes

Metadata:
{
  "processed_artworks": 2149,
  "dropped_records": 0,
  "classes": 8,
  "features": 128,
  "edges": 17093,
  "label_field": "artwork_type_title",
  "feature_fields": [
    "material_titles",
    "technique_titles",
    "place_of_origin",
    "medium_display"
  ],
  "optional_feature_coverage": {
    "subject_titles": 0.30153559795253604,
    "style_titles": 0.28757561656584457
  },
  "class_counts": {
    "Print": 395,
    "Drawing and Watercolor": 286,
    "Painting": 172,
    "Sculpture": 134,
    "Photograph": 386,
    "Architectural Drawing"

In [33]:

import os
import shutil

# Thư mục cần nén
source_dir = "/kaggle/working/Database"

# Đường dẫn file ZIP đầu ra
output_zip = "/kaggle/working/Database"

# Kiểm tra thư mục
if os.path.exists(source_dir):
    shutil.make_archive(output_zip, "zip", source_dir)

    zip_path = output_zip + ".zip"

    print("Nén dữ liệu thành công!")
    print(f"File ZIP: {zip_path}")
    print(f"Kích thước: {os.path.getsize(zip_path) / (1024**2):.2f} MB")
else:
    print(f"Không tìm thấy thư mục: {source_dir}")


Nén dữ liệu thành công!
File ZIP: /kaggle/working/Database.zip
Kích thước: 1.62 MB
